In [ ]:
from __future__ import annotations
from pathlib import Path
import os
import sys
sys.dont_write_bytecode = True
for variable in ('OMP_NUM_THREADS', 'OPENBLAS_NUM_THREADS', 'MKL_NUM_THREADS', 'VECLIB_MAXIMUM_THREADS'):
    os.environ.setdefault(variable, '1')
NOTEBOOK_NAME = '06_01_independent_roi_and_resting_connectivity.ipynb'
NOTEBOOK_RELATIVE_PATH = '06_01_independent_roi_and_resting_connectivity.ipynb'
WORKING_DIRECTORY = Path.cwd().resolve()
NOTEBOOK_CANDIDATES = [Path(os.environ['A8_NOTEBOOK_PATH']).expanduser()] if 'A8_NOTEBOOK_PATH' in os.environ else [WORKING_DIRECTORY / NOTEBOOK_NAME, WORKING_DIRECTORY / NOTEBOOK_RELATIVE_PATH]
NOTEBOOK_PATH = next((p.resolve() for p in NOTEBOOK_CANDIDATES if p.is_file()), None)
if NOTEBOOK_PATH is None:
    raise FileNotFoundError('Set A8_NOTEBOOK_PATH to this notebook, or run with its containing directory as the working directory')
ANALYSIS_ROOT = NOTEBOOK_PATH.parent


In [ ]:
from pathlib import Path as u09_Path
import sys as u09_sys
from IPython.display import display as u09_display, Image as u09_Image
import argparse as u09_argparse
import json as u09_json
from itertools import combinations as u09_combinations
import nibabel as u09_nib
import numpy as u09_np
import pandas as u09_pd
from nilearn import signal as u09_signal
u09_REST_EDGES = list(u09_combinations(range(5), 2))
u09_REST_COMPOSITES = {'shared_core': [(0, 1), (0, 2), (1, 2)], 'stroop_priority': [(i, 3) for i in range(3)], 'gonogo_priority': [(i, 4) for i in range(3)], 'bridge': [(3, 4)]}


In [ ]:
def u09_rest_confounds(path):
    cf = u09_pd.read_csv(path, sep=',' if u09_Path(path).suffix.lower() == '.csv' else '\t', dtype={'subject_id': str, 'subj_idx': str, 'run': str, 'trial_id': str})
    names = [name + s for name in u12_glm_MOTION_BASE + ['white_matter', 'csf'] for s in ('', '_derivative1', '_power2', '_derivative1_power2')]
    missing = set(names + ['framewise_displacement', 'std_dvars']) - set(cf.columns)
    if missing:
        raise ValueError(f'Missing confound regressors: {missing}')
    base = cf[names].apply(u09_pd.to_numeric, errors='raise').fillna(0).to_numpy()
    fd = cf.framewise_displacement.fillna(0).to_numpy()
    dv = cf.std_dvars.fillna(0).to_numpy()
    nss = cf.filter(regex='^non_steady_state_outlier').fillna(0).to_numpy()
    spikes = (fd > 0.5) | (dv > 1.5)
    if nss.shape[1]:
        spikes |= nss.any(axis=1)
    idx = u09_np.flatnonzero(spikes)
    regressors = u09_np.column_stack([base, u09_np.eye(len(cf))[:, idx]])
    qc = dict(mean_fd_mm=float(fd.mean()), fd_gt_05_fraction=float((fd > 0.5).mean()), all_spikes_fraction=float(spikes.mean()), n_spikes=len(idx), n_volumes=len(cf))
    qc['strict_motion'] = qc['mean_fd_mm'] < 0.2 and qc['fd_gt_05_fraction'] < 0.1 and (qc['all_spikes_fraction'] < 0.1)
    if qc['mean_fd_mm'] > 0.3 or qc['fd_gt_05_fraction'] > 0.2:
        raise ValueError('One or more of the 106 participants fail the prespecified resting-state motion criteria')
    return (regressors, qc)


In [ ]:
def u09_extract():
    masks = {r: u09_roi_masks(r) for r in u09_ROI_VARIANTS}
    cache = u09_REST_NEW
    cache.mkdir(parents=True, exist_ok=True)
    rows = []
    qcs = []
    for sid in u09_subjects():
        p = u12_glm_paths_for(sid, 'rest', None)
        (nuisance, qc) = u09_rest_confounds(p['confounds'])
        qcs.append(dict(subject_id=sid, **qc))
        fp = u09_fingerprint([p[k] for k in ('bold', 'mask', 'confounds', 'metadata')] + [u09_ROI_CONFIG, u09_Path(NOTEBOOK_PATH)], dict(version=u09_VERSION, nilearn=__import__('nilearn').__version__))
        dest = cache / f'{sid}_ROI_time_series.npz'
        if dest.exists():
            with u09_np.load(dest, allow_pickle=False) as z:
                cached = str(z['fingerprint']) == fp
                saved = {k: z[k].copy() for k in z.files} if cached else None
        else:
            cached = False
        if not cached:
            bold = u09_nib.load(p['bold'])
            brainim = u09_nib.load(p['mask'])
            ref = masks['ALE_3D'][1]
            if bold.shape[:3] != ref.shape or not u09_np.allclose(bold.affine, ref.affine):
                raise ValueError('Resting BOLD and ROI spaces do not match')
            if brainim.shape != ref.shape or not u09_np.allclose(brainim.affine, ref.affine):
                raise ValueError('Resting-state mask space does not match')
            (tr, _) = u12_glm_acquisition(p['metadata'])
            if bold.shape[-1] != len(nuisance):
                raise ValueError('Resting-state volume and confound counts do not match')
            data = u09_np.asarray(bold.dataobj, dtype='float32')
            brain = u09_np.asarray(brainim.dataobj) > 0
            valid = brain & u09_np.isfinite(data).all(axis=-1) & (data.std(axis=-1) > 0) & (abs(data.mean(axis=-1)) > 1e-06)
            gs = data[valid].mean(axis=0, dtype='float64')
            der = u09_np.r_[0.0, u09_np.diff(gs)]
            globalterms = u09_np.column_stack([gs, der, gs ** 2, der ** 2])
            saved = {'fingerprint': u09_np.asarray(fp)}
            for (radius, (roi, _)) in masks.items():
                coverage = u09_np.array([(valid & roi[r]).sum() / roi[r].sum() for r in u09_ROIS])
                eligible = coverage >= 0.8
                raw = u09_np.column_stack([data[valid & roi[r]].mean(axis=0) if eligible[i] else u09_np.zeros(data.shape[-1]) for (i, r) in enumerate(u09_ROIS)])
                saved[f'coverage_{radius}'] = coverage
                for gsr in (False, True):
                    cf = u09_np.column_stack([nuisance, globalterms]) if gsr else nuisance
                    cleaned = u09_signal.clean(raw, confounds=cf, detrend=True, standardize='zscore_sample', standardize_confounds=True, low_pass=0.08, high_pass=0.01, t_r=tr, ensure_finite=True)
                    cleaned[:, ~eligible] = u09_np.nan
                    saved[f'ts_{radius}_{int(gsr)}'] = cleaned
            del data
            temporary = dest.with_suffix('.tmp.npz')
            u09_np.savez_compressed(temporary, **saved)
            temporary.replace(dest)
        for radius in u09_ROI_VARIANTS:
            coverage = saved[f'coverage_{radius}']
            for gsr in (False, True):
                ts = saved[f'ts_{radius}_{int(gsr)}']
                z = u09_np.full((5, 5), u09_np.nan)
                for (i, j) in u09_REST_EDGES:
                    if coverage[i] >= 0.8 and coverage[j] >= 0.8:
                        if u09_np.std(ts[:, i]) == 0 or u09_np.std(ts[:, j]) == 0:
                            raise ValueError('Denoised time series has zero variance')
                        z[i, j] = u09_np.arctanh(u09_np.clip(u09_np.corrcoef(ts[:, i], ts[:, j])[0, 1], -1 + 1e-12, 1 - 1e-12))
                row = dict(subject_id=sid, roi_definition=radius, GSR=gsr, **qc)
                for (i, j) in u09_REST_EDGES:
                    row[u09_ROIS[i] + '__' + u09_ROIS[j]] = z[i, j]
                for (name, edges) in u09_REST_COMPOSITES.items():
                    row[name] = u09_np.mean([z[i, j] for (i, j) in edges])
                for (i, roi) in enumerate(u09_ROIS):
                    row['coverage_' + roi] = coverage[i]
                rows.append(row)
        print('Independent ALE ROI resting-state analysis complete', sid, flush=True)
    df = u09_pd.DataFrame(rows)
    df.to_csv(u09_REST_NEW / 'independent_ALE_ROI_connectivity_subject_level.csv', index=False)
    u09_pd.DataFrame(qcs).to_csv(u09_REST_NEW / 'observed_resting_motion.csv', index=False)
    u09_rest_summarize()


In [ ]:
def u09_rest_summarize():
    df = u09_pd.read_csv(u09_REST_NEW / 'independent_ALE_ROI_connectivity_subject_level.csv', dtype={'subject_id': str, 'subj_idx': str, 'run': str, 'trial_id': str})
    for ((radius, gsr), g) in df.groupby(['roi_definition', 'GSR']):
        suffix = f'{radius}_' + ('GSR' if gsr else 'noGSR')
        rows = []
        for (i, j) in u09_REST_EDGES:
            name = u09_ROIS[i] + '__' + u09_ROIS[j]
            rows.append(dict(edge=name, **u09_mean_stats(g[name])))
        out = u09_pd.DataFrame(rows)
        out['p_flip_FDR_10'] = u09_fdr(out.p_flip)
        u09_save_table(out, u09_REST, 'functional_connectivity_10_edges_' + suffix)
        comps = []
        for name in u09_REST_COMPOSITES:
            x = g[name].dropna()
            ci = u09_stats.t.interval(0.95, len(x) - 1, loc=x.mean(), scale=u09_stats.sem(x))
            comps.append(dict(composite=name, N=len(x), mean=x.mean(), SD=x.std(ddof=1), CI95_low=ci[0], CI95_high=ci[1]))
        u09_save_table(u09_pd.DataFrame(comps), u09_REST, 'four_prespecified_connectivity_measures_' + suffix)


In [ ]:
import matplotlib.pyplot as u09_plt
from matplotlib.text import Text as u09_Text
from matplotlib.ticker import FixedLocator as u09_FixedLocator, FixedFormatter as u09_FixedFormatter


In [ ]:
def u09_apa_plain_apa(text, bold=False):
    task_pattern = 'Go/No-Go|GoNoGo|Stroop|GNG(?: [12])?'

    def fragment(part):

        def task_math(match):
            cmd = 'mathbfit' if bold else 'mathit'
            word = match[0].replace(' ', chr(92) + ' ')
            pieces = word.split('-')
            return '$' + (chr(92) + 'text{-}').join((chr(92) + cmd + '{' + piece + '}' for piece in pieces)) + '$'
        part = u09_re.sub(task_pattern, task_math, part)
        tokens = u09_re.split('(\\$[^$]*\\$)', part)
        for (i, token) in enumerate(tokens):
            if not token.startswith('$'):
                tokens[i] = u09_re.sub('\\b(r|p|t|z|Z|N|n|M|SD|SE|df|d|F|v|a)\\b', lambda m: '$' + m[0] + '$', token)
        return ''.join(tokens)
    pieces = u09_re.split('(\\$[^$]*\\$)', text)
    return ''.join((part if part.startswith('$') else fragment(part) for part in pieces))


In [ ]:
def u09_apa_apply_apa_style(fig):
    if getattr(fig, '_a8_APA_applied', False):
        return
    fig._a8_APA_applied = True
    for axis in [axis for ax in fig.axes for axis in [ax.xaxis, ax.yaxis]]:
        current = [v.get_text() for v in axis.get_ticklabels()]
        changed = [u09_apa_plain_apa(v) for v in current]
        if current != changed:
            axis.set_major_locator(u09_FixedLocator(axis.get_majorticklocs()))
            axis.set_major_formatter(u09_FixedFormatter(changed))
    for text in fig.findobj(match=u09_Text):
        value = text.get_text()
        if not value:
            continue
        out = u09_apa_plain_apa(value, bold=text.get_fontweight() in ['bold', 'heavy', 700, 800, 900])
        if out != value:
            text.set_text(out)


In [ ]:
import re as u09_re
import numpy as u09_np
import pandas as u09_pd
from matplotlib.transforms import Bbox as u09_Bbox


In [ ]:
def u09_style_apply_final_style(fig, root, name):
    if getattr(fig, '_a8_final_style_applied', False):
        return
    fig._a8_final_style_applied = True
    ax = fig.axes[0]
    if name == 'Figure_S1_representative_design_matrix_n106':
        fig._a8_tight_export = True
    elif name == 'Figure_S1_head_motion_distribution_n106':
        for c in ax.collections:
            c.set_alpha(0.9)
            c.set_edgecolors('black')
            c.set_linewidths(0.25)
    elif name.startswith('Figure_S4_'):
        top = 304 if 'Stroop' in name else 219
        fig._a8_export_bbox = u09_Bbox.from_extents(150 / 600, 38 / 600, 4191 / 600, (2100 - top) / 600)
    elif name.startswith('Figure_S2_'):
        if 'individual_Dice_distribution' in name:
            for c in ax.collections:
                c.set_alpha(0.9)
                c.set_edgecolors('black')
                c.set_linewidths(0.25)
        elif 'activation_counts' in name:
            fig.legends[0].set_bbox_to_anchor((0.55, 0.06), transform=fig.transFigure)
    elif name.startswith('classical_brain_behavior_'):
        fig.set_size_inches(2.9, 1.9)
        rest = 'rest_connectivity' in name
        fig.subplots_adjust(left=0.37 if rest else 0.25, right=0.98, bottom=0.3, top=0.98)
        ax.xaxis.label.set_fontsize(6.5)
        ax.tick_params(labelsize=5.5)
        legend = fig.legends[0]
        legend.set_bbox_to_anchor((0.55, 0.01), transform=fig.transFigure)
        for text in legend.get_texts():
            text.set_fontsize(5.5)
        if rest:
            data = u09_pd.read_excel(u09_style_ROOT / '1.resting_state/tables/brain_behavior_4_connectivity_by_2_behavior_ALE_3D_noGSR.xlsx', dtype={'subject_id': str, 'subj_idx': str, 'run': str, 'trial_id': str})
            key = 'composite'
            order = ['shared_core', 'stroop_priority', 'gonogo_priority', 'bridge']
        else:
            data = u09_pd.read_excel(u09_style_ROOT / '2.task_fmri/tables/brain_behavior_5ROI_by_2task_by_2behavior_ALE_3D.xlsx', dtype={'subject_id': str, 'subj_idx': str, 'run': str, 'trial_id': str})
            data = data[data.task.eq('stroop' if 'Stroop' in name else 'gonogo')]
            key = 'roi_id'
            order = u09_ROIS
        for (k, outcome) in enumerate(['stroop_rt_cost_ms', 'nogo_accuracy']):
            for (j, (_, row)) in enumerate(data[data.outcome.eq(outcome)].set_index(key).loc[order].iterrows()):
                xpos = float(row.CI95_high) + 0.025
                if xpos < 0 and xpos + 0.15 > 0:
                    xpos = 0.025
                ax.text(xpos, j + (k - 0.5) * 0.27, f'{float(row.r):.3f}', ha='left', va='center', fontsize=5.2, color=['#3C5488', '#E64B35'][k])
    elif name.startswith('Figure_S7_'):
        fig.set_size_inches(3.6, 2.1)
        fig.subplots_adjust(left=0.2, right=0.985, bottom=0.2, top=0.89)
        ax.set_ylabel('Cross-task pattern\nsimilarity (Fisher z)', fontsize=7, labelpad=3)
        ax.tick_params(labelsize=6)
        for c in ax.collections:
            if len(c.get_offsets()) > 100:
                c.set_alpha(0.9)
                c.set_edgecolors('black')
                c.set_linewidths(0.25)
        data = u09_pd.read_excel(u09_style_ROOT / '2.task_fmri/tables/independent_ALE_ROI_ALE_3D_pattern_similarity_one_sided.xlsx', dtype={'subject_id': str, 'subj_idx': str, 'run': str, 'trial_id': str}).set_index('roi_id')
        for (label, key) in zip(ax.texts, u09_ROIS[:3]):
            val = float(data.loc[key, 'p_positive_FDR_3'])
            label.set_text('$p_{\\mathrm{FDR}} = ' + f'{val:.3f}' + '$')
            label.set_y(1.015)
            label.set_fontsize(7.5)
    elif name in ['Figure_independent_ALE_ROI_activation_n106', 'Figure_independent_ALE_ROI_cross_task_paired_difference_n106', 'Figure_shared_ALE_ROI_unsmoothed_pattern_similarity_n106']:
        base = u09_style_ROOT / '2.task_fmri/tables'
        if name == 'Figure_independent_ALE_ROI_activation_n106':
            one = u09_pd.read_excel(base / 'independent_ALE_ROI_ALE_3D_10_activation_tests.xlsx', dtype={'subject_id': str, 'subj_idx': str, 'run': str, 'trial_id': str})
            for (k, task) in enumerate(u09_CONTRAST):
                for (j, (_, row)) in enumerate(one[one.task.eq(task)].set_index('roi_id').loc[u09_ROIS].iterrows()):
                    if row.p_flip_FDR_10 < 0.05:
                        ax.text(float(row.CI95_high) + 0.025, j + (k - 0.5) * 0.22, '*', fontsize=8, color=['#397A9E', '#B76B3F'][k], ha='left', va='center')
            (lo, hi) = ax.get_xlim()
            ax.set_xlim(lo, hi + 0.06)
        else:
            paired = 'paired' in name
            data = u09_pd.read_excel(base / ('independent_ALE_ROI_ALE_3D_cross_task_paired_tests.xlsx' if paired else 'independent_ALE_ROI_ALE_3D_pattern_similarity_one_sided.xlsx'), dtype={'subject_id': str, 'subj_idx': str, 'run': str, 'trial_id': str}).set_index('roi_id')
            for (j, key) in enumerate(u09_ROIS if paired else u09_ROIS[:3]):
                q = float(data.loc[key, 'p_FDR_5' if paired else 'p_positive_FDR_3'])
                value = '< 0.001' if q < 0.001 else '= ' + (f'{q:.5f}' if 0.05 < q < 0.0505 else f'{q:.3f}')
                ax.text(0.98, j - 0.26, '$p_{\\mathrm{FDR}} ' + value + '$', transform=ax.get_yaxis_transform(), ha='right', va='center', fontsize=7.5)
    elif name in ['Figure_independent_ALE_ROI_resting_connectivity_matrix_n106', 'Figure_independent_ALE_ROI_four_prespecified_connectivity_measures_n106']:
        for text in list(fig.texts):
            text.remove()
        fig._a8_tight_export = True
        if 'matrix' in name:
            fig.set_size_inches(2.8, 2.5)
            fig.subplots_adjust(left=0.21, right=0.81, bottom=0.25, top=0.93)
            fig.axes[1].set_position([0.84, 0.25, 0.025, 0.68])
            fig.axes[1].yaxis.label.set_fontsize(6.5)
            fig.axes[1].tick_params(labelsize=5.5)
            edges = u09_pd.read_excel(u09_style_ROOT / '1.resting_state/tables/functional_connectivity_10_edges_ALE_3D_noGSR.xlsx', dtype={'subject_id': str, 'subj_idx': str, 'run': str, 'trial_id': str})
            lookup = {frozenset(row.edge.split('__')): row for (_, row) in edges.iterrows()}
            for text in ax.texts:
                (j, i) = map(int, text.get_position())
                row = lookup[frozenset([u09_ROIS[i], u09_ROIS[j]])]
                text.set_text(f"{float(row['mean']):.2f}" + ('*' if row.p_flip_FDR_10 < 0.05 else ''))
                text.set_fontsize(6.5)
            ax.tick_params(labelsize=6, length=2)
        else:
            fig.set_size_inches(2.9, 1.9)
            fig.subplots_adjust(left=0.38, right=0.98, bottom=0.23, top=0.98)
            ax.xaxis.label.set_fontsize(6.5)
            ax.tick_params(labelsize=6, length=2)


In [ ]:
from functools import lru_cache as u09_lru_cache
import tempfile as u09_tempfile
import hashlib as u09_hashlib
import json as u09_json
import warnings as u09_warnings
import numpy as u09_np
import nibabel as u09_nib
import matplotlib as u09_matplotlib
import matplotlib.pyplot as u09_plt
from matplotlib.colors import Normalize as u09_Normalize, ListedColormap as u09_ListedColormap, BoundaryNorm as u09_BoundaryNorm, LinearSegmentedColormap as u09_LinearSegmentedColormap
from nilearn import datasets as u09_datasets, surface as u09_surface, plotting as u09_plotting
u09_brain_POSITIVE_STOPS = [(0.0, '#B2182B'), (0.45, '#E64B35'), (0.75, '#F39B2F'), (1.0, '#FDE34F')]
u09_brain_BRAIN_POSITIVE_CMAP = u09_LinearSegmentedColormap.from_list('a8_positive', u09_brain_POSITIVE_STOPS, N=4096)
u09_brain_BRAIN_SIGNED_CMAP = u09_LinearSegmentedColormap.from_list('a8_signed', [(0.0, '#55C9DD'), (0.25, '#237BC0'), (0.49999, '#24489E'), (0.5, '#F7F7F7')] + [(0.50001 + 0.49999 * x, c) for (x, c) in u09_brain_POSITIVE_STOPS], N=8192)
u09_brain_ROI_COLORS = ['#E64B35', '#4DBBD5', '#00A087', '#3C5488', '#F39B7F']
u09_brain_ROI_LABELS = ['SMA', 'rAI/FO', 'lAI', 'lIPS', 'rIFG/PMC']
u09_brain_VIEWS = [('left', 'lateral', 'L lateral'), ('left', 'medial', 'L medial'), ('right', 'medial', 'R medial'), ('right', 'lateral', 'R lateral')]


In [ ]:
def u09_brain_anatomical_shading(ax, mesh_path, values, sulc, hemi, view, categorical=False):
    (faces, luminance) = u09_brain_cortical_luminance(str(mesh_path), str(sulc), hemi, view)
    fv = values[faces]
    active = u09_np.median(fv, axis=1) > 0.5 if categorical else (u09_np.all(fv > 0, axis=1) | u09_np.all(fv < 0, axis=1)) & (u09_np.abs(fv.mean(axis=1)) >= 1e-08)
    changed = 0
    for coll in ax.collections:
        original = coll._facecolor3d.copy()
        if len(original) != len(faces):
            continue
        result = original.copy()
        result[~active, :3] = luminance[~active, None]
        assert u09_np.array_equal(result[active], original[active])
        assert u09_np.array_equal(result[:, 3], original[:, 3])
        edges = coll._edgecolor3d.copy()
        assert edges.shape == original.shape
        edges[~active, :3] = luminance[~active, None]
        assert u09_np.array_equal(edges[active], coll._edgecolor3d[active])
        coll.set_facecolors(result)
        coll.set_edgecolors(edges)
        changed += 1
    assert changed == 1


In [ ]:
def u09_brain_strict_face_mean(values):
    active = u09_np.all(values > 0) | u09_np.all(values < 0)
    return float(values.mean()) if active else 0.0


In [ ]:
def u09_brain_project(img, mask, categorical=False):
    h = u09_hashlib.sha256()
    for im in (img, mask):
        h.update(u09_np.asanyarray(im.dataobj).tobytes())
        h.update(im.affine.tobytes())
    h.update(u09_source_signature().encode())
    h.update(u09_sha(u09_input_path('MNI152_display_reference.nii.gz')).encode())
    h.update(str(categorical).encode())
    h.update(u09_sha(u09_brain_inverse_transform()).encode())
    key = h.hexdigest()
    dest = u09_brain_CACHE / (key + '.npz')
    if dest.exists():
        with u09_np.load(dest, allow_pickle=False) as z:
            return {k: z[k].copy() for k in z.files}
    import ants as ants
    fs = u09_brain_meshes()
    with u09_tempfile.TemporaryDirectory(prefix='a8_surface_') as tmp:
        tmp = u09_Path(tmp)
        u09_nib.save(img, tmp / 'values.nii.gz')
        u09_nib.save(mask, tmp / 'mask.nii.gz')
        fixed = ants.image_read(str(u09_input_path('MNI152_display_reference.nii.gz')))
        transform = [str(u09_brain_inverse_transform())]
        warped = ants.apply_transforms(fixed=fixed, moving=ants.image_read(str(tmp / 'values.nii.gz')), transformlist=transform, interpolator='genericLabel' if categorical else 'linear')
        valid = ants.apply_transforms(fixed=fixed, moving=ants.image_read(str(tmp / 'mask.nii.gz')), transformlist=transform, interpolator='genericLabel')
        ants.image_write(warped, str(tmp / 'values6.nii.gz'))
        ants.image_write(valid, str(tmp / 'mask6.nii.gz'))
        result = {}
        for hemi in ('left', 'right'):
            with u09_warnings.catch_warnings():
                u09_warnings.filterwarnings('ignore', message='Mean of empty slice')
                values = u09_surface.vol_to_surf(str(tmp / 'values6.nii.gz'), fs['pial_' + hemi], inner_mesh=fs['white_' + hemi], kind='depth', depth=[0.5], interpolation='nearest_most_frequent' if categorical else 'linear', mask_img=None if categorical else str(tmp / 'mask6.nii.gz'))
            result[hemi] = u09_np.nan_to_num(values, nan=0.0)
            if categorical:
                assert set(u09_np.unique(result[hemi])).issubset(set(range(6)))
    temp = dest.with_suffix('.tmp.npz')
    u09_np.savez_compressed(temp, **result)
    temp.replace(dest)
    (u09_brain_CACHE / (key + '.json')).write_text(u09_json.dumps(dict(purpose='visualization only', template_transform=str(u09_brain_inverse_transform().name), transform_sha256=u09_sha(u09_brain_inverse_transform()), native_space='MNI152NLin2009cAsym', surface_frame='MNI152NLin6Asym', mesh='fsaverage6; mid-ribbon samples', depth=0.5, categorical=categorical, nonzero_vertices={k: int(u09_np.count_nonzero(v)) for (k, v) in result.items()}), indent=2))
    return result


In [ ]:
def u09_brain_surface_row(fig, rect, textures, cmap, vmin, vmax, categorical=False, labels=True):
    fs = u09_brain_meshes()
    (x, y, w, h) = rect
    gap = 0.012 * w
    aw = (w - 3 * gap) / 4
    axes = []
    for (i, (hemi, view, name)) in enumerate(u09_brain_VIEWS):
        ax = fig.add_axes([x + i * (aw + gap), y, aw, h], projection='3d')
        opts = dict(surf_mesh=fs['pial_' + hemi], bg_map=fs['sulc_' + hemi], hemi=hemi, view=view, engine='matplotlib', cmap=cmap, colorbar=False, vmin=vmin, vmax=vmax, bg_on_data=not categorical and vmin == 0 and (vmax == 1), alpha=1, axes=ax, figure=fig)
        if categorical:
            u09_plotting.plot_surf_roi(roi_map=textures[hemi], avg_method='median', threshold=0.5, **opts)
        else:
            u09_plotting.plot_surf_stat_map(stat_map=textures[hemi], threshold=1e-08, symmetric_cbar=vmin < 0, avg_method=u09_brain_strict_face_mean, **opts)
        u09_brain_anatomical_shading(ax, fs['pial_' + hemi], textures[hemi], fs['sulc_' + hemi], hemi, view, categorical)
        ax.set_box_aspect(None, zoom=1.5)
        for collection in ax.collections:
            collection.set_rasterized(True)
        if labels:
            fig.text(x + (i + 0.5) * (aw + gap) - gap / 2, y + 0.012, name, ha='center', va='center', fontsize=7)
        axes.append(ax)
    return axes


In [ ]:
def u09_brain_axial_row(fig, rect, img, cuts, cmap, norm, categorical=False):
    im = u09_nib.as_closest_canonical(img)
    bg = u09_brain_background()
    assert im.shape == bg.shape and u09_np.allclose(im.affine, bg.affine), 'Axial image/background grids differ'
    values = u09_np.asarray(im.dataobj)
    anat = u09_np.where(u09_brain_background_mask(), u09_np.asarray(bg.dataobj), 0.0)
    shape = im.shape
    xyz0 = u09_nib.affines.apply_affine(im.affine, [-0.5, -0.5, 0])
    xyz1 = u09_nib.affines.apply_affine(im.affine, [shape[0] - 0.5, shape[1] - 0.5, 0])
    extent = [xyz0[0], xyz1[0], xyz0[1], xyz1[1]]
    vmax_bg = float(u09_np.quantile(anat[anat > 0], 0.995))
    (x, y, w, h) = rect
    gap = 0.018 * w
    aw = (w - gap * (len(cuts) - 1)) / len(cuts)
    axes = []
    for (i, z) in enumerate(cuts):
        ax = fig.add_axes([x + i * (aw + gap), y, aw, h])
        k = int(round(u09_nib.affines.apply_affine(u09_np.linalg.inv(im.affine), [0, 0, z])[2]))
        a = anat[:, :, k].T
        v = values[:, :, k].T
        ax.imshow(u09_np.ma.masked_less_equal(a, 0), origin='lower', extent=extent, cmap='gray', vmin=0, vmax=vmax_bg, interpolation='bilinear')
        shown = u09_np.ma.masked_where(abs(v) < 1e-09, v)
        ax.imshow(shown, origin='lower', extent=extent, cmap=cmap, norm=norm, interpolation='nearest')
        ax.set(xlim=(-80, 80), ylim=(-112, 80))
        ax.set_axis_off()
        fig.text(x + i * (aw + gap) + aw / 2, y - 0.015, f'z = {im.affine[2, 3] + k * im.affine[2, 2]:g} mm', fontsize=7, ha='center', va='center')
        if i == 0:
            ax.text(0.03, 0.94, 'L', transform=ax.transAxes, fontsize=7, va='top')
            ax.text(0.91, 0.94, 'R', transform=ax.transAxes, fontsize=7, va='top')
        axes.append(ax)
    return axes


In [ ]:
def u09_brain_set_alignment(fig, rows):
    axes = []
    ids = []
    groups = []
    for (i, row) in enumerate(rows):
        names = [f'row{i}_view{j}' for j in range(len(row))]
        axes += row
        ids += names
        groups.append(names)
    fig._a8_alignment_options = dict(axes=axes, panel_ids=ids, row_groups=groups, column_groups=[])


In [ ]:
def u09_brain_rows(fig, upper, lower, img, cmap=u09_brain_BRAIN_SIGNED_CMAP, vmin=-6, vmax=6, cuts=(-20, 6, 20, 40, 60), categorical=False, annotations=None):
    data = u09_np.asarray(img.dataobj)
    mask = u09_nib.Nifti1Image((abs(data) > 1e-09).astype('uint8'), img.affine)
    tex = u09_brain_project(img, mask, categorical=categorical)
    top = u09_brain_surface_row(fig, upper, tex, cmap, vmin, vmax, categorical=categorical)
    norm = u09_BoundaryNorm(u09_np.arange(0.5, 6, 1), 5) if categorical else u09_Normalize(vmin, vmax)
    bottom = u09_brain_axial_row(fig, lower, img, cuts, cmap, norm, categorical=categorical)
    if annotations is not None:
        u09_annotation_annotate_rows(fig, top, bottom, img, cuts, tex, categorical, annotations)
    return [top, bottom]


In [ ]:
def u09_brain_horizontal_bar(fig, rect, cmap, vmin, vmax, label='Z statistic'):
    cax = fig.add_axes(rect)
    cb = fig.colorbar(u09_plt.cm.ScalarMappable(norm=u09_Normalize(vmin, vmax), cmap=cmap), cax=cax, orientation='horizontal', extend='both' if vmin < 0 else 'max')
    cb.set_ticks(u09_np.linspace(vmin, vmax, 5 if vmin < 0 else 4))
    from matplotlib.ticker import FormatStrFormatter as FormatStrFormatter
    cb.ax.xaxis.set_major_formatter(FormatStrFormatter('%.1f'))
    cb.ax.tick_params(labelsize=6, length=2, width=0.5, pad=1)
    fig.text(rect[0] + rect[2] + 0.025, rect[1] + rect[3] / 2, label, fontsize=7, va='center')
    return cb


In [ ]:
def u09_brain_stat_figure(img, title, subtitle, cmap=u09_brain_BRAIN_SIGNED_CMAP, vmin=-6, vmax=6, label_scope='positive'):
    fig = u09_plt.figure(figsize=(7.2, 3.5))
    panel_rows = u09_brain_rows(fig, (0.01, 0.5, 0.98, 0.4), (0.01, 0.13, 0.98, 0.29), img, cmap, vmin, vmax, annotations=label_scope)
    fig.text(0.5, 0.965, title, fontsize=10, fontweight='bold', va='top', ha='center')
    u09_brain_horizontal_bar(fig, (0.3, 0.055, 0.38, 0.02), cmap, vmin, vmax, label='Minimum Z' if vmin == 0 else 'Z statistic')
    fig._a8_caption = subtitle
    u09_brain_set_alignment(fig, panel_rows)
    return fig


In [ ]:
from functools import lru_cache as u09_lru_cache
import numpy as u09_np
import nibabel as u09_nib
from mpl_toolkits.mplot3d import proj3d as u09_proj3d
u09_annotation_LABELS = {1: 'SMA', 2: 'rAI/FO', 3: 'lAI', 4: 'lIPS', 5: 'rIFG/PMC'}
u09_annotation_FULL_NAMES = {1: 'supplementary motor area', 2: 'right anterior insula / frontal operculum', 3: 'left anterior insula', 4: 'left intraparietal sulcus', 5: 'right inferior frontal gyrus / premotor cortex'}


In [ ]:
@u09_lru_cache(maxsize=1)
def u09_annotation_roi_display():
    (masks, ref) = u09_roi_masks()
    labels = u09_np.zeros(ref.shape, dtype='uint8')
    for (i, r) in enumerate(u09_ROIS, 1):
        labels[masks[r]] = i
    img = u09_nib.Nifti1Image(labels, ref.affine)
    mask = u09_nib.Nifti1Image((labels > 0).astype('uint8'), ref.affine)
    return (masks, ref, u09_brain_project(img, mask, categorical=True))


In [ ]:
def u09_annotation_not_occluded(index, projected, faces):
    triangles = projected[faces]
    p = triangles[index].mean(0)
    box = (triangles[:, :, :2].min(1) <= p[:2]).all(1) & (triangles[:, :, :2].max(1) >= p[:2]).all(1)
    ids = u09_np.flatnonzero(box)
    ids = ids[ids != index]
    if not len(ids):
        return True
    tri = triangles[ids]
    a = tri[:, 0, :2]
    v = tri[:, 1, :2] - a
    u = tri[:, 2, :2] - a
    r = p[:2] - a
    det = v[:, 0] * u[:, 1] - v[:, 1] * u[:, 0]
    good = u09_np.abs(det) > 1e-12
    s = u09_np.divide(r[:, 0] * u[:, 1] - r[:, 1] * u[:, 0], det, out=u09_np.zeros(len(ids)), where=good)
    t = u09_np.divide(v[:, 0] * r[:, 1] - v[:, 1] * r[:, 0], det, out=u09_np.zeros(len(ids)), where=good)
    inside = good & (s >= -1e-07) & (t >= -1e-07) & (s + t <= 1 + 1e-07)
    depth = tri[:, 0, 2] + s * (tri[:, 1, 2] - tri[:, 0, 2]) + t * (tri[:, 2, 2] - tri[:, 0, 2])
    return not u09_np.any(inside & (depth < p[2] - 1e-05))


In [ ]:
def u09_annotation_arrow(ax, text, xy, pos, ha, xycoords='data'):
    return ax.annotate(text, xy=xy, xycoords=xycoords, xytext=pos, textcoords='axes fraction', fontsize=6.5, color='#161616', ha=ha, va='top', annotation_clip=False, arrowprops=dict(arrowstyle='->', lw=0.8, color='#161616', mutation_scale=6, shrinkA=2.5, shrinkB=0))


In [ ]:
def u09_annotation_surface_label(ax, values, roi_texture, idx, hemi, view):
    (xyz, faces) = u09_surface.load_surf_mesh(u09_brain_meshes()['pial_' + hemi])
    xyz = u09_np.asarray(xyz, dtype=float)
    fv = values[faces]
    member = (roi_texture[faces] == idx).all(1)
    candidates = u09_np.flatnonzero(member & (fv > 0).all(1))
    if not len(candidates):
        return None
    points = xyz[faces]
    normal = u09_np.cross(points[:, 1] - points[:, 0], points[:, 2] - points[:, 0])
    if u09_np.median(u09_np.sum(normal * (points.mean(1) - xyz.mean(0)), axis=1)) < 0:
        normal = -normal
    (az, el) = u09_np.deg2rad([ax.azim, ax.elev])
    camera = u09_np.array([u09_np.cos(el) * u09_np.cos(az), u09_np.cos(el) * u09_np.sin(az), u09_np.sin(el)])
    candidates = candidates[normal[candidates] @ camera > 0]
    centred = xyz - xyz.mean(0)
    projected = u09_np.column_stack(u09_proj3d.proj_transform(*centred.T, ax.get_proj()))
    order = candidates[u09_np.argsort(-fv[candidates].mean(1), kind='stable')]
    chosen = None
    for face in order[:1000]:
        if u09_annotation_not_occluded(int(face), projected, faces):
            chosen = int(face)
            break
    if chosen is None:
        return None
    anchor = projected[faces[chosen], :2].mean(0)
    pos = (0.94, 0.97) if hemi == 'right' else (0.06, 0.97)
    u09_annotation_arrow(ax, u09_annotation_LABELS[idx], anchor, pos, 'right' if hemi == 'right' else 'left')
    return dict(roi_id=u09_ROIS[idx - 1], label=u09_annotation_LABELS[idx], full_name=u09_annotation_FULL_NAMES[idx], panel='surface', hemisphere=hemi, view=view, face_index=chosen, vertices=faces[chosen].tolist(), displayed_value=float(fv[chosen].mean()), label_membership='All three vertices inside independently projected ALE ROI', visibility='Camera-facing; projected triangle centroid passes occlusion test', anchor_definition='Visible positive face within ROI, not whole-brain cluster peak')


In [ ]:
def u09_annotation_axial_label(axes, img, cuts, masks, idx, categorical=False):
    data = u09_np.asarray(img.dataobj)
    mask = masks[u09_ROIS[idx - 1]]
    valid = mask & (data == idx if categorical else data > 0)
    choices = []
    for (i, z) in enumerate(cuts):
        k = int(round(u09_nib.affines.apply_affine(u09_np.linalg.inv(img.affine), [0, 0, z])[2]))
        plane = valid[:, :, k]
        if plane.any():
            choices.append((i, k, int(plane.sum())))
    if not choices:
        return None
    preferred = [c for c in choices if c[0] == 1] if idx in (2, 3) else []
    (i, k, _) = max(preferred or choices, key=lambda c: c[2])
    ij = u09_np.argwhere(valid[:, :, k])
    v = data[ij[:, 0], ij[:, 1], k]
    if categorical:
        from scipy.ndimage import distance_transform_edt as distance_transform_edt
        dist = distance_transform_edt(valid[:, :, k])
        scores = dist[ij[:, 0], ij[:, 1]]
    else:
        scores = v
    point = ij[int(u09_np.argmax(scores))]
    ijk = (int(point[0]), int(point[1]), int(k))
    assert mask[ijk] and (data[ijk] == idx if categorical else data[ijk] > 0)
    xyz = u09_nib.affines.apply_affine(img.affine, ijk)
    if idx == 1:
        pos = (0.5, 1.14)
        ha = 'center'
    elif xyz[0] < 0:
        pos = (0.02, 1.14)
        ha = 'left'
    else:
        pos = (0.98, 1.14)
        ha = 'right'
    u09_annotation_arrow(axes[i], u09_annotation_LABELS[idx], xyz[:2], pos, ha)
    return dict(roi_id=u09_ROIS[idx - 1], label=u09_annotation_LABELS[idx], full_name=u09_annotation_FULL_NAMES[idx], panel='axial', slice_index=i, voxel_ijk=list(ijk), MNI2009c_xyz_mm=xyz.tolist(), displayed_value=float(data[ijk]), label_membership='Original independent ALE mask contains voxel', anchor_definition='Displayed positive voxel inside ROI on an existing slice; not necessarily cluster peak')


In [ ]:
def u09_annotation_annotate_rows(fig, top, bottom, img, cuts, textures, categorical=False, scope='positive'):
    (masks, ref, roi_texture) = u09_annotation_roi_display()
    assert img.shape == ref.shape and u09_np.allclose(img.affine, ref.affine)
    allowed = (1, 2, 3) if scope == 'shared' else (1, 2, 3, 4, 5)
    records = []
    done = set()
    for (idx, view_index, hemi, view) in [(4, 0, 'left', 'lateral'), (1, 1, 'left', 'medial'), (5, 3, 'right', 'lateral')]:
        if idx not in allowed:
            continue
        record = u09_annotation_surface_label(top[view_index], textures[hemi], roi_texture[hemi], idx, hemi, view)
        if record is not None:
            records.append(record)
            done.add(idx)
    for idx in allowed:
        if idx in done:
            continue
        record = u09_annotation_axial_label(bottom, img, cuts, masks, idx, categorical)
        if record is not None:
            records.append(record)
            done.add(idx)
    if categorical:
        assert done == set(allowed), f'Independent ROI labels missing: {set(allowed) - done}'
    for r in records:
        r['map_role'] = 'independent_ROI_definition' if categorical else 'displayed_positive_map_within_independent_ROI'
    fig._a8_annotations = getattr(fig, '_a8_annotations', []) + records
    return records


In [ ]:
import os as u09_os
import subprocess as u09_subprocess
import matplotlib as u09_matplotlib
u09_matplotlib.use('Agg')
import matplotlib.pyplot as u09_plt
from matplotlib.lines import Line2D as u09_Line2D
import numpy as u09_np
import pandas as u09_pd
u09_plt.rcParams.update({'font.family': 'Arial', 'font.size': 7, 'axes.spines.top': False, 'axes.spines.right': False, 'axes.linewidth': 0.7, 'pdf.fonttype': 42, 'svg.fonttype': 'none', 'xtick.labelsize': 6, 'ytick.labelsize': 7, 'legend.frameon': False})
u09_PLOT_LABELS = ['SMA', 'rAI/FO', 'lAI', 'lIPS', 'rIFG/PMC']
u09_PLOT_COLORS = {'stroop': '#397A9E', 'gonogo': '#B76B3F'}
u09_PLOT_TASK_FIGURE_NAMES = ['Figure_independent_ALE_ROI_activation_n106', 'Figure_independent_ALE_ROI_cross_task_paired_difference_n106', 'Figure_shared_ALE_ROI_unsmoothed_pattern_similarity_n106']
u09_PLOT_QC_FIGURE_NAMES = ['Figure_S1_representative_design_matrix_n106', 'Figure_S1_head_motion_distribution_n106']
u09_PLOT_COVERAGE_FIGURE_NAMES = ['Figure_S1_Stroop_standard_space_brain_coverage_n106', 'Figure_S1_GoNoGo_standard_space_brain_coverage_n106']
u09_PLOT_MAXT_FIGURE_NAMES = ['Figure_S4_Stroop_voxelwise_maxT_FWE_n106', 'Figure_S4_GoNoGo_voxelwise_maxT_FWE_n106']


In [ ]:
def u09_export(fig, root, name):
    u09_style_apply_final_style(fig, root, name)
    u09_apa_apply_apa_style(fig)
    out = root / 'image'
    out.mkdir(exist_ok=True)
    fig.canvas.draw()
    fig.savefig(out / (name + '.png'), dpi=600, facecolor='white', bbox_inches='tight')
    u09_plt.close(fig)


In [ ]:
def u09_plot_rest():
    base = u09_REST / 'tables'
    edges = u09_pd.read_excel(base / 'functional_connectivity_10_edges_ALE_3D_noGSR.xlsx', dtype={'subject_id': str, 'subj_idx': str, 'run': str, 'trial_id': str})
    comps = u09_pd.read_excel(base / 'four_prespecified_connectivity_measures_ALE_3D_noGSR.xlsx', dtype={'subject_id': str, 'subj_idx': str, 'run': str, 'trial_id': str}).set_index('composite').loc[['shared_core', 'stroop_priority', 'gonogo_priority', 'bridge']]
    mat = u09_np.full((5, 5), u09_np.nan)
    for (_, r) in edges.iterrows():
        (a, b) = r.edge.split('__')
        (i, j) = (u09_ROIS.index(a), u09_ROIS.index(b))
        mat[i, j] = mat[j, i] = r['mean']
    (fig, ax) = u09_plt.subplots(figsize=(3.8, 3.4))
    fig.subplots_adjust(left=0.19, right=0.79, bottom=0.27, top=0.94)
    vmax = max(0.1, float(u09_np.nanmax(u09_np.abs(mat))))
    im = ax.imshow(mat, cmap='RdBu_r', vmin=-vmax, vmax=vmax, aspect='auto')
    ax.set(xticks=range(5), yticks=range(5), xticklabels=u09_PLOT_LABELS, yticklabels=u09_PLOT_LABELS)
    u09_plt.setp(ax.get_xticklabels(), rotation=40, rotation_mode='anchor', ha='right')
    for i in range(5):
        for j in range(5):
            if i != j:
                ax.text(j, i, f'{mat[i, j]:.2f}', ha='center', va='center', fontsize=7, color='white' if mat[i, j] > vmax * 0.6 else '#253746')
    cax = fig.add_axes([0.83, 0.27, 0.025, 0.67])
    cb = fig.colorbar(im, cax=cax)
    cb.set_label('Mean Fisher z', fontsize=7)
    cb.ax.tick_params(labelsize=6)
    fig.text(0.19, 0.055, f'No GSR; N = {int(edges.N.min())}–{int(edges.N.max())}.', fontsize=6)
    fig._a8_alignment_options = dict(axes=[ax], exclude_axes=[cax])
    assert not ax.get_title() and (not any((t.get_text() in {'a', 'b'} for t in ax.texts)))
    u09_export(fig, u09_REST, 'Figure_independent_ALE_ROI_resting_connectivity_matrix_n106')
    (fig, ax) = u09_plt.subplots(figsize=(3.8, 3.4))
    fig.subplots_adjust(left=0.36, right=0.95, bottom=0.25, top=0.94)
    ax.errorbar(comps['mean'], u09_np.arange(4), xerr=u09_np.array([comps['mean'] - comps.CI95_low, comps.CI95_high - comps['mean']]), fmt='o', color='#397A9E', ms=4, lw=0.9, capsize=2)
    ax.set(yticks=u09_np.arange(4), yticklabels=['Shared core', 'Stroop priority', 'Go/No-Go priority', 'Bridge'], ylim=(3.6, -0.6), xlabel='Mean Fisher z')
    ax.axvline(0, color='.7', lw=0.6, ls='--', zorder=0)
    fig.text(0.12, 0.055, f'No GSR; N = {int(comps.N.min())}–{int(comps.N.max())}. Means and 95% CIs.', fontsize=6)
    assert not ax.get_title() and (not ax.texts)
    u09_export(fig, u09_REST, 'Figure_independent_ALE_ROI_four_prespecified_connectivity_measures_n106')


In [ ]:
def u09_roi_definition():
    from matplotlib.colors import ListedColormap as ListedColormap
    from matplotlib.patches import Patch as Patch
    (masks, ref) = u09_roi_masks()
    labels = u09_np.zeros(ref.shape, dtype='uint8')
    occupied = u09_np.zeros(ref.shape, bool)
    for (i, r) in enumerate(u09_ROIS, 1):
        assert not (occupied & masks[r]).any(), 'Categorical display requires disjoint masks'
        labels[masks[r]] = i
        occupied |= masks[r]
    im = u09_nib.Nifti1Image(labels, ref.affine)
    cmap = ListedColormap(u09_brain_ROI_COLORS)
    fig = u09_plt.figure(figsize=(7.2, 3.5))
    panel_rows = u09_brain_rows(fig, (0.01, 0.5, 0.98, 0.4), (0.01, 0.13, 0.98, 0.29), im, cmap=cmap, vmin=0.5, vmax=5.5, cuts=(0, 6, 36, 42, 50), categorical=True, annotations='roi_definition')
    fig.text(0.5, 0.965, 'Five independent ALE regions', fontsize=10, fontweight='bold', va='top', ha='center')
    fig.legend([Patch(facecolor=c) for c in u09_brain_ROI_COLORS], u09_brain_ROI_LABELS, loc='lower center', bbox_to_anchor=(0.5, 0.005), ncol=5, fontsize=7, handlelength=1.2, columnspacing=1.7)
    u09_brain_set_alignment(fig, panel_rows)
    u09_export(fig, u09_REST, 'Supplementary_resting_ALE_inhibitory_control_ROIs_surface_and_axial_5_regions')


In [ ]:
from pathlib import Path as u12_Path
import json as u12_json
import os as u12_os
from pathlib import Path as u12_Path
import numpy as u12_np
from pathlib import Path as u12_Path
u12_glm_TR = 2.0
u12_glm_MOTION_BASE = ['trans_x', 'trans_y', 'trans_z', 'rot_x', 'rot_y', 'rot_z']


In [ ]:
def u12_glm_paths_for(subject, task, run=None):
    return u12_imaging_paths(subject, task, run)


In [ ]:
def u12_glm_acquisition(path: u12_Path) -> tuple[float, float]:
    meta = u12_json.loads(path.read_text(encoding='utf-8'))
    tr = float(meta.get('RepetitionTime', u12_glm_TR))
    start = float(meta.get('StartTime', 0.0))
    if not u12_np.isclose(tr, u12_glm_TR):
        raise ValueError(f'Unexpected TR={tr}: {path}')
    return (tr, start / tr)


In [ ]:
import json as u12_json
import os as u12_os
from pathlib import Path as u12_Path
import matplotlib as u12_matplotlib
import matplotlib.pyplot as u12_plt
from matplotlib import font_manager as u12_font_manager
import numpy as u12_np
import json as u12_json
import numpy as u12_np
import matplotlib.pyplot as u12_plt
import numpy as u12_np
import json as u12_json
import numpy as u12_np
import matplotlib as u12_matplotlib
import matplotlib.pyplot as u12_plt
import numpy as u12_np
import os as u12_os
import matplotlib as u12_matplotlib
u12_matplotlib.use('Agg')
import matplotlib.pyplot as u12_plt
import numpy as u12_np
u12_plt.rcParams.update({'font.family': 'Arial', 'font.size': 7, 'axes.spines.top': False, 'axes.spines.right': False, 'axes.linewidth': 0.7, 'pdf.fonttype': 42, 'svg.fonttype': 'none', 'xtick.labelsize': 6, 'ytick.labelsize': 7, 'legend.frameon': False})
from pathlib import Path as u13_Path
import json as u13_json
import nibabel as u13_nib
import numpy as u13_np
import pandas as u13_pd
import json as u13_json
import subprocess as u13_subprocess
import tempfile as u13_tempfile
import nibabel as u13_nib
import numpy as u13_np
import pandas as u13_pd


In [ ]:
def u13_roi_prepare_roi(source, reference, destination, transforms=()):
    (source, reference, destination) = map(u13_Path, (source, reference, destination))
    im = u13_nib.load(source)
    a = u13_np.asarray(im.dataobj)
    if not u13_np.isin(a, [0, 1]).all() or not a.any():
        raise ValueError('Source ROI must be nonempty and binary')
    ref = u13_nib.load(reference)
    transform_paths = [u13_Path(p) for p in transforms]
    if any((not p.is_file() for p in transform_paths)):
        raise FileNotFoundError('An ANTs transform file is missing')
    destination.parent.mkdir(parents=True, exist_ok=True)
    with u13_tempfile.TemporaryDirectory(prefix='a8_roi_') as temp:
        if transform_paths:
            output = u13_Path(temp) / 'mapped.nii.gz'
            command = ['antsApplyTransforms', '-d', '3', '-i', str(source), '-r', str(reference), '-o', str(output), '-n', 'GenericLabel']
            for transform in transform_paths:
                command.extend(['-t', str(transform)])
            u13_subprocess.run(command, check=True, capture_output=True, text=True)
            im = u13_nib.load(output)
        a = u13_np.asarray(im.dataobj)
        if im.shape != ref.shape or not u13_np.allclose(im.affine, ref.affine):
            raise ValueError('ROI must match the analysis grid; supply the required ANTs transforms')
        if not u13_np.isin(a, [0, 1]).all() or not a.any():
            raise ValueError('Mapped ROI must be nonempty and binary')
        u13_nib.save(im, destination)
    return u13_nib.load(destination)


In [ ]:
def u13_roi_define():
    table = u15_table
    file_ref = u15_file_ref
    input_path = u15_input_path
    src = table('ROI_masks.csv', ['roi_id', 'path', 'source_doi', 'independent_of_sample'], ['roi_id'])
    if set(src.roi_id) != set(u13_ROIS):
        raise ValueError('Need the five documented independent ROI masks')
    if not src.independent_of_sample.astype(str).str.lower().isin(['true', '1']).all():
        raise ValueError('ROI independence must be explicitly documented')
    reference = u13_first_path(u13_subjects()[0], 'stroop', 'effect')
    u13_NEW.mkdir(parents=True, exist_ok=True)
    manifest = {}
    rows = []
    cfg = {'source_doi': src.source_doi.tolist(), 'input_manifest_sha256': u13_sha(input_path('ROI_masks.csv')), 'target_space': 'MNI152NLin2009cAsym'}
    u13_atomic_json(u13_ROI_CONFIG, cfg)
    for row in src.itertuples():
        source = file_ref(row.path)
        raw_transforms = getattr(row, 'transforms', None)
        transforms = [] if u13_pd.isna(raw_transforms) or not str(raw_transforms).strip() else u13_json.loads(raw_transforms)
        if not isinstance(transforms, list) or not all((isinstance(p, str) for p in transforms)):
            raise ValueError('transforms must be a JSON array of paths in ANTs CLI order')
        transforms = [file_ref(p) for p in transforms]
        dest = u13_NEW / f'{row.roi_id}_ALE_3D.nii.gz'
        im = u13_roi_prepare_roi(source, reference, dest, transforms)
        a = u13_np.asarray(im.dataobj)
        xyz = u13_nib.affines.apply_affine(im.affine, u13_np.argwhere(a > 0)).mean(axis=0)
        rows.append(dict(roi_id=row.roi_id, target_voxels=int(a.sum()), volume_mm3=float(a.sum() * abs(u13_np.linalg.det(im.affine[:3, :3]))), centroid_x=xyz[0], centroid_y=xyz[1], centroid_z=xyz[2], source=str(source), source_sha256=u13_sha(source), reference_doi=row.source_doi))
        manifest[row.roi_id] = {'sha256': u13_sha(dest), 'source_sha256': u13_sha(source), 'reference_sha256': u13_sha(reference), 'transforms_cli_order': [{'path': str(p), 'sha256': u13_sha(p)} for p in transforms], 'interpolation': 'GenericLabel' if transforms else 'already_on_target_grid'}
    u13_atomic_json(u13_NEW / 'definition_and_provenance.json', {'config_sha256': u13_sha(u13_ROI_CONFIG), 'masks': manifest, 'selection_independent_of_current_sample': True})
    u13_save_table(u13_pd.DataFrame(rows), u13_TASK, 'independent_ALE_3D_ROI_definition_5_regions')
    return u13_pd.DataFrame(rows)


In [ ]:
import matplotlib.pyplot as u13_plt
import numpy as u13_np
import pandas as u13_pd
import subprocess as u13_subprocess
import matplotlib as u13_matplotlib
u13_matplotlib.use('Agg')
import matplotlib.pyplot as u13_plt
import numpy as u13_np
import pandas as u13_pd
import numpy as u13_np
import pandas as u13_pd
import matplotlib.pyplot as u13_plt
from pathlib import Path as u15_Path
import os as u15_os, hashlib as u15_hashlib, json as u15_json
import numpy as u15_np
import pandas as u15_pd
u15_ROOT = ANALYSIS_ROOT
u15_DATA = u15_Path(u15_os.environ.get('A8_DATA_DIR', NOTEBOOK_PATH.parent)).expanduser().resolve()
u15_RESULTS = u15_Path(u15_os.environ.get('A8_OUTPUT_DIR', u15_ROOT / 'results')).expanduser().resolve()
u15_EXPECTED_N = int(u15_os.environ.get('A8_EXPECTED_N', '106'))
u15_PARAMS = ['stroop_a', 'stroop_v_congruent', 'stroop_v_neutral', 'stroop_v_incongruent', 'stroop_t', 'gonogo_a', 'gonogo_v_go', 'gonogo_v_nogo', 'gonogo_t', 'gonogo_z']
u15_INPUT_FOLDERS = {'MRI_manifest.csv': '7.task_fmri', 'Participants.csv': '1.behavior', 'ROI_masks.csv': '7.task_fmri', 'HDDM_posterior_draws.csv': '2.hddm_fitting', 'HDDM_parameters.csv': '2.hddm_fitting', 'Behavior_data.csv': '1.behavior', 'HDDM_data.csv': '2.hddm_fitting'}
u15_INPUT_FOLDERS.update({'MRI_preprocessing.csv': '7.task_fmri', 'MultiEcho_runs.csv': '7.task_fmri'})
u15_INPUT_FILES = {name: name for name in u15_INPUT_FOLDERS}
u15_INPUT_COLUMNS = {'Participants.csv': ('subject_id', 'age', 'sex', 'education'), 'Behavior_data.csv': ('subject_id', 'task', 'run', 'trial_id', 'condition', 'rt_s', 'response', 'correct'), 'HDDM_data.csv': ('subj_idx', 'task', 'run', 'trial_id', 'condition', 'stimulus', 'response', 'rt'), 'MRI_manifest.csv': ('subject_id', 'task', 'run', 'bold', 'mask', 'confounds', 'metadata', 'events'), 'ROI_masks.csv': ('roi_id', 'path', 'source_doi', 'independent_of_sample'), 'HDDM_parameters.csv': ('subject_id', *u15_PARAMS), 'HDDM_posterior_draws.csv': ('task', 'chain', 'draw', 'subject_id', 'parameter', 'value')}


In [ ]:
def u15_input_path(name):
    filename = u15_INPUT_FILES.get(name, name)
    p = u15_DATA / filename
    if not p.is_file() and 'A8_DATA_DIR' not in u15_os.environ:
        p = u15_ROOT / u15_INPUT_FOLDERS.get(name, '7.task_fmri') / filename
    if name == 'MRI_manifest.csv' and (u15_os.environ.get('A8_MRI_INPUT_MODE') == 'generated' or not p.is_file()):
        p = u15_RESULTS / '7.task_fmri/tables/MRI_manifest.csv'
    fitted = (u15_RESULTS / '2.hddm_fitting/fit_state.json').is_file() and u15_os.environ.get('A8_HDDM_INPUT_MODE', 'auto') != 'external'
    if name in {'HDDM_parameters.csv', 'HDDM_posterior_draws.csv'} and (fitted or not p.is_file()):
        p = u15_RESULTS / '2.hddm_fitting/tables' / name
    if not p.is_file():
        raise FileNotFoundError(f'Required observed input missing: {p}. Supply the required real input file; no example values are substituted.')
    return p


In [ ]:
def u15_table(name, columns=(), unique=None):
    p = u15_input_path(name)
    d = u15_pd.read_csv(p, dtype={'subject_id': str, 'subj_idx': str, 'run': str, 'trial_id': str})
    if d.empty:
        raise ValueError(f'{p.name}: no observed records; a header-only template is not data')
    missing = (set(columns) | set(u15_INPUT_COLUMNS.get(name, ()))) - set(d)
    if missing:
        raise ValueError(f'{p.name}: missing columns {sorted(missing)}')
    if unique and (d[list(unique)].isna().any().any() or d.duplicated(list(unique)).any()):
        raise ValueError(f'{p.name}: missing/duplicate key {unique}')
    for c in ('subject_id', 'subj_idx', 'trial_id'):
        if c in d and (d[c].isna().any() or d[c].str.strip().eq('').any() or d[c].ne(d[c].str.strip()).any()):
            raise ValueError(f'{p.name}: blank or whitespace-padded identifier in {c}')
    for c in ('data_status', 'source_status'):
        if c in d and d[c].astype(str).str.contains('synthetic|simulation|not empirical', case=False, regex=True).any():
            raise ValueError(f'{p.name}: simulated rows cannot enter this workflow')
    return d


In [ ]:
def u15_participants():
    d = u15_table('Participants.csv', ['subject_id', 'age', 'sex', 'education'], ['subject_id'])
    if len(d) != u15_EXPECTED_N:
        raise ValueError(f'Expected {u15_EXPECTED_N} participants, found {len(d)}; configure A8_EXPECTED_N deliberately.')
    for c in ['age', 'sex', 'education']:
        d[c] = u15_pd.to_numeric(d[c], errors='raise')
        if not u15_np.isfinite(d[c]).all():
            raise ValueError(f'Nonfinite {c}')
    if not d.sex.isin([0, 1]).all():
        raise ValueError('sex must use the documented 0/1 coding')
    return d.sort_values('subject_id').reset_index(drop=True)


In [ ]:
def u15_match_cohort(d, name, keys=('subject_id',), complete=True):
    ids = set(u15_participants().subject_id)
    if d.subject_id.isna().any() or not set(d.subject_id) <= ids:
        raise ValueError(f'{name}: unexpected or missing subject_id')
    if complete and set(d.subject_id) != ids:
        raise ValueError(f'{name}: cohort mismatch; do not fill missing participants')
    if d.duplicated(list(keys)).any():
        raise ValueError(f'{name}: duplicate {keys}')
    return d


In [ ]:
def u15_file_ref(value):
    if u15_pd.isna(value) or not str(value).strip():
        raise ValueError('Empty input path')
    p = u15_Path(str(value)).expanduser()
    p = p if p.is_absolute() else u15_DATA / p
    if not p.is_file():
        raise FileNotFoundError(p)
    return p.resolve()


In [ ]:
def u15_imaging_manifest():
    d = u15_table('MRI_manifest.csv', ['subject_id', 'task', 'run', 'bold', 'mask', 'confounds', 'metadata', 'events'])
    d['run'] = d.run.fillna('').str.replace('\\.0$', '', regex=True).str.lstrip('0').replace({'': 'single'})
    u15_match_cohort(d, 'MRI_manifest', ('subject_id', 'task', 'run'))
    if not d.task.isin(['stroop', 'gonogo', 'rest']).all():
        raise ValueError('Invalid MRI task')
    return d


In [ ]:
def u15_imaging_paths(subject, task, run=None):
    run = str(run or 'single').lstrip('0') or 'single'
    d = u15_imaging_manifest()
    g = d[(d.subject_id == subject) & (d.task == task) & (d.run == run)]
    if len(g) != 1:
        raise ValueError(f'Expected one MRI run: {subject}/{task}/{run}, found {len(g)}')
    row = g.iloc[0]
    cols = ['bold', 'mask', 'confounds', 'metadata'] + ([] if task == 'rest' else ['events'])
    return {c: u15_file_ref(row[c]) for c in cols}


In [ ]:
def u15_source_signature():
    document = u15_json.loads(NOTEBOOK_PATH.read_text())
    source = '\n'.join((''.join(cell['source']) for cell in document['cells'] if cell['cell_type'] == 'code'))
    return u15_hashlib.sha256(source.encode()).hexdigest()


In [ ]:
from pathlib import Path as u17__A8Path
from pathlib import Path as u17_Path
import sys as u17_sys
import hashlib as u17_hashlib
import json as u17_json
import os as u17_os
for u17__key in ('OMP_NUM_THREADS', 'OPENBLAS_NUM_THREADS', 'VECLIB_MAXIMUM_THREADS'):
    u17_os.environ.setdefault(u17__key, '1')
import nibabel as u17_nib
import numpy as u17_np
import pandas as u17_pd
import nilearn as u17_nilearn
if u17_nilearn.__version__ != '0.13.1':
    raise RuntimeError('Use Nilearn 0.13.1, as specified in the manuscript')
from scipy import stats as u17_stats
from statsmodels.stats.multitest import multipletests as u17_multipletests
import sys as u17_sys
from pathlib import Path as u17_Path
u17_PROJECT = u15_ROOT
u17_RESULTS = u15_RESULTS
u17_DATA = u15_DATA
u17_EXPECTED_N = u15_EXPECTED_N
u17_participants = u15_participants
u17_source_signature = u15_source_signature
u17_MRI = u17_RESULTS / '3.mri_analysis'
u17_TASK = u17_MRI / '2.task_fmri'
u17_REST = u17_MRI / '1.resting_state'
u17_VERSION = 'observed_inputs_v1'
u17_LEGACY = u17_DATA / 'ale_masks'
u17_NEW = u17_TASK / 'other_outputs'
u17_REST_NEW = u17_REST / 'other_outputs'
u17_ROI_CONFIG = u17_MRI / 'ROI_definition_ALE_3D_clusters.json'
u17_ROI_VARIANTS = ('ALE_3D',)
u17_ROIS = ['shared_SMA', 'shared_rAI_FO', 'shared_lAI', 'stroop_lIPS', 'gonogo_rIFG_PMC']
u17_CONTRAST = {'stroop': 'incongruent_minus_congruent', 'gonogo': 'nogo_correct_minus_go_correct'}
u17_N_PERM = 50000
u17_SEED = 20261017


In [ ]:
def u17_subjects():
    return u17_participants().subject_id.tolist()


In [ ]:
def u17_sha(path):
    h = u17_hashlib.sha256()
    with u17_Path(path).open('rb') as f:
        for block in iter(lambda : f.read(1024 * 1024), b''):
            h.update(block)
    return h.hexdigest()


In [ ]:
def u17_fingerprint(paths, extra=None):
    info = [(str(u17_Path(p).resolve()), u17_Path(p).stat().st_size, u17_Path(p).stat().st_mtime_ns) for p in paths]
    return u17_hashlib.sha256(u17_json.dumps([info, extra, u17_source_signature()], sort_keys=True).encode()).hexdigest()


In [ ]:
def u17_atomic_json(path, data):
    path = u17_Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix('.tmp')

    def native(value):
        if isinstance(value, u17_np.generic):
            return value.item()
        if isinstance(value, u17_np.ndarray):
            return value.tolist()
        raise TypeError(f'Unsupported metadata type: {type(value).__name__}')
    tmp.write_text(u17_json.dumps(data, ensure_ascii=False, indent=2, allow_nan=False, default=native))
    tmp.replace(path)


In [ ]:
def u17_save_table(frame, root, name):
    if '10_activation_tests' in name or name.startswith('functional_connectivity_10_edges_'):
        frame = frame.drop(columns=['p_flip_positive'], errors='ignore')
    if name.startswith('functional_connectivity_10_edges_'):
        frame = frame.drop(columns=['t', 'df', 'p_t'], errors='ignore')
    if 'pattern_similarity_' in name:
        frame = frame.drop(columns=['t', 'df', 'p_t'], errors='ignore')
    dest = root / 'tables'
    dest.mkdir(parents=True, exist_ok=True)
    path = dest / f'{name}.xlsx'
    frame.to_excel(path, index=False)
    from openpyxl import load_workbook as load_workbook
    from openpyxl.styles import Font as Font, Border as Border, Side as Side, Alignment as Alignment
    from openpyxl.utils import get_column_letter as get_column_letter
    wb = load_workbook(path)
    ws = wb.active
    rule = Side(style='thin', color='333333')
    ws.sheet_view.showGridLines = False
    ws.freeze_panes = 'A2'
    for (col, key) in enumerate(frame.columns, 1):
        letter = get_column_letter(col)
        ws.column_dimensions[letter].width = min(38, max(13, len(str(key)) + 2))
        cell = ws.cell(1, col)
        cell.font = Font(name='Arial', size=10, bold=True)
        cell.alignment = Alignment(wrap_text=True, vertical='center')
        cell.border = Border(top=rule, bottom=rule)
        for row in range(2, ws.max_row + 1):
            c = ws.cell(row, col)
            c.font = Font(name='Arial', size=10)
            if isinstance(c.value, str):
                c.alignment = Alignment(wrap_text=True, vertical='top')
                visual_length = sum((2 if ord(char) > 255 else 1 for char in c.value))
                lines = max(1, int(u17_np.ceil(visual_length / ws.column_dimensions[letter].width)))
                ws.row_dimensions[row].height = max(ws.row_dimensions[row].height or 15, min(150, lines * 15))
            if isinstance(c.value, float):
                c.number_format = '0.000E+00' if str(key).startswith(('p', 'q')) else '0.000'
            if row == ws.max_row:
                c.border = Border(bottom=rule)
    ws.row_dimensions[1].height = 32
    wb.save(path)


In [ ]:
def u17_fdr(p):
    p = u17_np.asarray(p, float)
    out = u17_np.full(p.shape, u17_np.nan)
    valid = u17_np.isfinite(p)
    if valid.any():
        out[valid] = u17_multipletests(p[valid], method='fdr_bh')[1]
    return out


In [ ]:
def u17_mean_stats(x, seed=u17_SEED, alternative='two-sided'):
    if alternative not in ('two-sided', 'greater'):
        raise ValueError('Unsupported alternative')
    key = 'p_flip' if alternative == 'two-sided' else 'p_flip_positive'
    x = u17_np.asarray(x, float)
    x = x[u17_np.isfinite(x)]
    n = len(x)
    if n < 3:
        return dict(N=n, mean=u17_np.nan, SD=u17_np.nan, t=u17_np.nan, df=n - 1, CI95_low=u17_np.nan, CI95_high=u17_np.nan, **{key: u17_np.nan})
    mu = x.mean()
    rng = u17_np.random.default_rng(seed)
    exceed = 0
    for start in range(0, u17_N_PERM, 1000):
        signs = rng.choice((-1.0, 1.0), size=(min(1000, u17_N_PERM - start), n))
        perm = signs @ x / n
        exceed += u17_np.count_nonzero(abs(perm) >= abs(mu) - 1e-14) if alternative == 'two-sided' else u17_np.count_nonzero(perm >= mu - 1e-14)
    sem = u17_stats.sem(x)
    ci = u17_stats.t.interval(0.95, n - 1, loc=mu, scale=sem)
    return dict(N=n, mean=float(mu), SD=float(x.std(ddof=1)), t=float(mu / sem), df=n - 1, CI95_low=float(ci[0]), CI95_high=float(ci[1]), **{key: (exceed + 1) / (u17_N_PERM + 1)})


In [ ]:
def u17_first_path(sid, task, stat):
    name = f'{sid}_task-{task}_contrast-{u17_CONTRAST[task]}_stat-{stat}.nii.gz'
    fresh = u17_NEW / name
    if not u17_new_first_complete(sid, task):
        raise FileNotFoundError(f'Current first-level GLM is incomplete; fallback to previous input results is prohibited: {sid}/{task}')
    return fresh


In [ ]:
def u17_new_first_complete(sid, task):
    suffix = 'stroop' if task == 'stroop' else 'gonogo_duration500ms'
    qc = u17_NEW / f'{sid}_task-{suffix}_first_level_qc.csv'
    maps = [u17_NEW / f'{sid}_task-{task}_contrast-{u17_CONTRAST[task]}_stat-{stat}.nii.gz' for stat in ('effect', 'variance', 'z')]
    maps.append(u17_NEW / f'{sid}_task-{task}_mask-analysis.nii.gz')
    return qc.exists() and all((p.exists() for p in maps))


In [ ]:
def u17_qc_path(sid, task):
    suffix = 'stroop' if task == 'stroop' else 'gonogo_duration500ms'
    name = f'{sid}_task-{suffix}_first_level_qc.csv'
    fresh = u17_NEW / name
    if not u17_new_first_complete(sid, task):
        raise FileNotFoundError(f'Current first-level GLM is incomplete: {sid}/{task}')
    return fresh


In [ ]:
def u17_t_map(sid, task):
    e = u17_nib.load(u17_first_path(sid, task, 'effect'))
    v = u17_nib.load(u17_first_path(sid, task, 'variance'))
    if e.shape != v.shape or not u17_np.allclose(e.affine, v.affine):
        raise ValueError('Effect and variance image spaces do not match')
    (ed, vd) = (u17_np.asarray(e.dataobj, float), u17_np.asarray(v.dataobj, float))
    valid = u17_np.isfinite(ed) & u17_np.isfinite(vd) & (vd > 0)
    t = u17_np.divide(ed, u17_np.sqrt(u17_np.maximum(vd, 0)), out=u17_np.zeros_like(ed), where=valid)
    qc = u17_pd.read_csv(u17_qc_path(sid, task), dtype={'subject_id': str, 'subj_idx': str, 'run': str, 'trial_id': str})
    dof = int((qc.n_timepoints - qc.design_rank).sum())
    return (t, valid, e, dof)


In [ ]:
def u17_roi_masks(definition='ALE_3D'):
    if definition != 'ALE_3D':
        raise ValueError('The specified analysis uses independent three-dimensional ALE cluster masks only')
    cfg = u17_json.loads(u17_ROI_CONFIG.read_text())
    manifest = u17_NEW / 'definition_and_provenance.json'
    if not manifest.exists():
        raise FileNotFoundError('Provide and validate independent ALE ROI inputs, or run the ROI definition step in this notebook')
    meta = u17_json.loads(manifest.read_text())
    if meta['config_sha256'] != u17_sha(u17_ROI_CONFIG):
        raise ValueError('ROI definition changed; regenerate ROI masks')
    ref = u17_nib.load(u17_first_path(u17_subjects()[0], 'stroop', 'effect'))
    masks = {}
    for roi in u17_ROIS:
        path = u17_NEW / f'{roi}_ALE_3D.nii.gz'
        if u17_sha(path) != meta['masks'][roi]['sha256']:
            raise ValueError('ALE mask validation failed')
        img = u17_nib.load(path)
        data = u17_np.asarray(img.dataobj)
        if img.shape != ref.shape or not u17_np.allclose(img.affine, ref.affine):
            raise ValueError('ALE ROI and subject statistic image spaces do not match')
        if not u17_np.isin(data, [0, 1]).all() or not data.any():
            raise ValueError('ALE mask must be nonempty and binary')
        masks[roi] = data > 0
    return (masks, ref)


In [ ]:
u09_ROOT = u15_ROOT
u09_DATA = u15_DATA
u09_RESULTS = u15_RESULTS
u09__A8Path = u17__A8Path
u09_Path = u17_Path
u09_sys = u17_sys
u09_hashlib = u17_hashlib
u09_json = u17_json
u09_os = u17_os
u09_nib = u17_nib
u09_np = u17_np
u09_pd = u17_pd
u09_nilearn = u17_nilearn
u09_stats = u17_stats
u09_multipletests = u17_multipletests
u09_PROJECT = u17_PROJECT
u09_RESULTS = u17_RESULTS
u09_DATA = u17_DATA
u09_EXPECTED_N = u17_EXPECTED_N
u09_participants = u17_participants
u09_source_signature = u17_source_signature
u09_MRI = u17_MRI
u09_TASK = u17_TASK
u09_REST = u17_REST
u09_VERSION = u17_VERSION
u09_LEGACY = u17_LEGACY
u09_NEW = u17_NEW
u09_REST_NEW = u17_REST_NEW
u09_ROI_CONFIG = u17_ROI_CONFIG
u09_ROI_VARIANTS = u17_ROI_VARIANTS
u09_ROIS = u17_ROIS
u09_CONTRAST = u17_CONTRAST
u09_N_PERM = u17_N_PERM
u09_SEED = u17_SEED
u09_subjects = u17_subjects
u09_sha = u17_sha
u09_fingerprint = u17_fingerprint
u09_atomic_json = u17_atomic_json
u09_save_table = u17_save_table
u09_fdr = u17_fdr
u09_mean_stats = u17_mean_stats
u09_first_path = u17_first_path
u09_new_first_complete = u17_new_first_complete
u09_qc_path = u17_qc_path
u09_t_map = u17_t_map
u09_roi_masks = u17_roi_masks
u09__A8Path = u17__A8Path
u09_Path = u17_Path
u09_sys = u17_sys
u09_hashlib = u17_hashlib
u09_json = u17_json
u09_os = u17_os
u09_nib = u17_nib
u09_np = u17_np
u09_pd = u17_pd
u09_nilearn = u17_nilearn
u09_stats = u17_stats
u09_multipletests = u17_multipletests
u09_PROJECT = u17_PROJECT
u09_RESULTS = u17_RESULTS
u09_DATA = u17_DATA
u09_EXPECTED_N = u17_EXPECTED_N
u09_participants = u17_participants
u09_source_signature = u17_source_signature
u09_MRI = u17_MRI
u09_TASK = u17_TASK
u09_REST = u17_REST
u09_VERSION = u17_VERSION
u09_LEGACY = u17_LEGACY
u09_NEW = u17_NEW
u09_REST_NEW = u17_REST_NEW
u09_ROI_CONFIG = u17_ROI_CONFIG
u09_ROI_VARIANTS = u17_ROI_VARIANTS
u09_ROIS = u17_ROIS
u09_CONTRAST = u17_CONTRAST
u09_N_PERM = u17_N_PERM
u09_SEED = u17_SEED
u09_subjects = u17_subjects
u09_sha = u17_sha
u09_fingerprint = u17_fingerprint
u09_atomic_json = u17_atomic_json
u09_save_table = u17_save_table
u09_fdr = u17_fdr
u09_mean_stats = u17_mean_stats
u09_first_path = u17_first_path
u09_new_first_complete = u17_new_first_complete
u09_qc_path = u17_qc_path
u09_t_map = u17_t_map
u09_roi_masks = u17_roi_masks
u09_plt.rcParams.update({'mathtext.fontset': 'custom', 'mathtext.rm': 'Arial', 'mathtext.it': 'Arial:italic', 'mathtext.bf': 'Arial:bold', 'mathtext.bfit': 'Arial:italic:bold', 'mathtext.sf': 'Arial'})
u09_ROIS = u17_ROIS
u09_CONTRAST = u17_CONTRAST
u09_MRI = u17_MRI
u09_matplotlib.use('Agg')
u09_MRI = u17_MRI
u09_REST = u17_REST
u09_LEGACY = u17_LEGACY
u09_NEW = u17_NEW
u09_ROIS = u17_ROIS
u09_sha = u17_sha
u09_input_path = u15_input_path
u09_source_signature = u15_source_signature


In [ ]:
@u09_lru_cache(maxsize=1)
def u09_brain_meshes():
    return u09_datasets.fetch_surf_fsaverage(mesh='fsaverage6')


In [ ]:
@u09_lru_cache(maxsize=1)
def u09_brain_background():
    return u09_nib.as_closest_canonical(u09_nib.load(u09_input_path('Template_T1w.nii.gz')))


In [ ]:
@u09_lru_cache(maxsize=1)
def u09_brain_background_mask():
    path = u09_Path.home() / '.cache/templateflow/tpl-MNI152NLin2009cAsym/tpl-MNI152NLin2009cAsym_res-02_desc-brain_mask.nii.gz'
    if not path.is_file():
        from templateflow.api import get as get
        path = u09_Path(get('MNI152NLin2009cAsym', resolution=2, desc='brain', suffix='mask', extension='.nii.gz'))
    mask = u09_nib.as_closest_canonical(u09_nib.load(path))
    bg = u09_brain_background()
    assert mask.shape == bg.shape and u09_np.allclose(mask.affine, bg.affine)
    return u09_np.asarray(mask.dataobj) > 0


In [ ]:
@u09_lru_cache(maxsize=8)
def u09_brain_cortical_luminance(mesh_path, sulc_path, hemi, view):
    (xyz, faces) = u09_surface.load_surf_mesh(mesh_path)
    xyz = u09_np.asarray(xyz, dtype=float)
    faces = u09_np.asarray(faces)
    points = xyz[faces]
    area_normal = u09_np.cross(points[:, 1] - points[:, 0], points[:, 2] - points[:, 0])
    if u09_np.median(u09_np.sum(area_normal * (points.mean(axis=1) - xyz.mean(axis=0)), axis=1)) < 0:
        area_normal = -area_normal
    vertex_normal = u09_np.zeros_like(xyz)
    for corner in range(3):
        u09_np.add.at(vertex_normal, faces[:, corner], area_normal)
    vertex_normal /= u09_np.maximum(u09_np.linalg.norm(vertex_normal, axis=1, keepdims=True), 1e-12)
    normal = vertex_normal[faces].mean(axis=1)
    normal /= u09_np.maximum(u09_np.linalg.norm(normal, axis=1, keepdims=True), 1e-12)
    side = -1 if (hemi == 'left') == (view == 'lateral') else 1
    key = u09_np.array([side * 0.8, -0.45, 0.75])
    key /= u09_np.linalg.norm(key)
    fill = u09_np.array([side * 0.6, 0.65, 0.2])
    fill /= u09_np.linalg.norm(fill)
    illumination = 0.55 + 0.38 * u09_np.maximum(normal @ key, 0) + 0.07 * u09_np.maximum(normal @ fill, 0)
    sulc = u09_np.asarray(u09_surface.load_surf_data(sulc_path))
    (lo, hi) = u09_np.percentile(sulc, [2, 98])
    curvature = u09_np.clip((sulc - lo) / (hi - lo), 0, 1)[faces].mean(axis=1)
    luminance = (0.94 - 0.18 * curvature) * illumination
    assert u09_np.isfinite(luminance).all() and u09_np.all((luminance >= 0) & (luminance <= 1))
    return (faces, luminance)


In [ ]:
@u09_lru_cache(maxsize=1)
def u09_brain_inverse_transform():
    p = u09_input_path('MNI2009c_to_MNI152_display_transform.h5')
    if not p.is_file() or p.stat().st_size < 1000:
        raise FileNotFoundError('TemplateFlow MNI2009c-to-MNI152 display transform is required')
    return p


In [ ]:
u09_ROIS = u17_ROIS
u09_roi_masks = u17_roi_masks
u09__A8Path = u17__A8Path
u09_Path = u17_Path
u09_sys = u17_sys
u09_hashlib = u17_hashlib
u09_json = u17_json
u09_os = u17_os
u09_nib = u17_nib
u09_np = u17_np
u09_pd = u17_pd
u09_nilearn = u17_nilearn
u09_stats = u17_stats
u09_multipletests = u17_multipletests
u09_PROJECT = u17_PROJECT
u09_RESULTS = u17_RESULTS
u09_DATA = u17_DATA
u09_EXPECTED_N = u17_EXPECTED_N
u09_participants = u17_participants
u09_source_signature = u17_source_signature
u09_MRI = u17_MRI
u09_TASK = u17_TASK
u09_REST = u17_REST
u09_VERSION = u17_VERSION
u09_LEGACY = u17_LEGACY
u09_NEW = u17_NEW
u09_REST_NEW = u17_REST_NEW
u09_ROI_CONFIG = u17_ROI_CONFIG
u09_ROI_VARIANTS = u17_ROI_VARIANTS
u09_ROIS = u17_ROIS
u09_CONTRAST = u17_CONTRAST
u09_N_PERM = u17_N_PERM
u09_SEED = u17_SEED
u09_subjects = u17_subjects
u09_sha = u17_sha
u09_fingerprint = u17_fingerprint
u09_atomic_json = u17_atomic_json
u09_save_table = u17_save_table
u09_fdr = u17_fdr
u09_mean_stats = u17_mean_stats
u09_first_path = u17_first_path
u09_new_first_complete = u17_new_first_complete
u09_qc_path = u17_qc_path
u09_t_map = u17_t_map
u09_roi_masks = u17_roi_masks
u12_imaging_paths = u15_imaging_paths
u12_matplotlib.use('Agg')
u12_Path = u17_Path
u12_json = u17_json
u12_os = u17_os
u12_np = u17_np
u12_MRI = u17_MRI
u12_Path = u17_Path
u12_json = u17_json
u12_os = u17_os
u12_np = u17_np
u12_MRI = u17_MRI
u12_plt.rcParams.update({'mathtext.fontset': 'custom', 'mathtext.rm': 'Arial', 'mathtext.it': 'Arial:italic', 'mathtext.bf': 'Arial:bold', 'mathtext.bfit': 'Arial:italic:bold', 'mathtext.sf': 'Arial'})
u12_MRI = u17_MRI
u12_matplotlib.use('Agg')
u12_MRI = u17_MRI
u12_Path = u17_Path
u12_json = u17_json
u12_os = u17_os
u12_np = u17_np
u12_MRI = u17_MRI
u13_Path = u17_Path
u13_json = u17_json
u13_nib = u17_nib
u13_np = u17_np
u13_pd = u17_pd
u13_TASK = u17_TASK
u13_NEW = u17_NEW
u13_ROI_CONFIG = u17_ROI_CONFIG
u13_ROIS = u17_ROIS
u13_subjects = u17_subjects
u13_sha = u17_sha
u13_atomic_json = u17_atomic_json
u13_save_table = u17_save_table
u13_first_path = u17_first_path
u13_Path = u17_Path
u13_json = u17_json
u13_nib = u17_nib
u13_np = u17_np
u13_pd = u17_pd
u13_TASK = u17_TASK
u13_NEW = u17_NEW
u13_ROI_CONFIG = u17_ROI_CONFIG
u13_ROIS = u17_ROIS
u13_subjects = u17_subjects
u13_sha = u17_sha
u13_atomic_json = u17_atomic_json
u13_save_table = u17_save_table
u13_first_path = u17_first_path
u13_Path = u17_Path
u13_json = u17_json
u13_nib = u17_nib
u13_np = u17_np
u13_pd = u17_pd
u13_TASK = u17_TASK
u13_NEW = u17_NEW
u13_ROI_CONFIG = u17_ROI_CONFIG
u13_ROIS = u17_ROIS
u13_subjects = u17_subjects
u13_sha = u17_sha
u13_atomic_json = u17_atomic_json
u13_save_table = u17_save_table
u13_first_path = u17_first_path
u13_plt.rcParams.update({'mathtext.fontset': 'custom', 'mathtext.rm': 'Arial', 'mathtext.it': 'Arial:italic', 'mathtext.bf': 'Arial:bold', 'mathtext.bfit': 'Arial:italic:bold', 'mathtext.sf': 'Arial'})
u13_ROIS = u17_ROIS
u13_Path = u17_Path
u13_json = u17_json
u13_nib = u17_nib
u13_np = u17_np
u13_pd = u17_pd
u13_TASK = u17_TASK
u13_NEW = u17_NEW
u13_ROI_CONFIG = u17_ROI_CONFIG
u13_ROIS = u17_ROIS
u13_subjects = u17_subjects
u13_sha = u17_sha
u13_atomic_json = u17_atomic_json
u13_save_table = u17_save_table
u13_first_path = u17_first_path
u13_plt.rcParams.update({'font.family': 'Arial', 'font.size': 7, 'axes.spines.top': False, 'axes.spines.right': False, 'axes.linewidth': 0.7, 'pdf.fonttype': 42, 'svg.fonttype': 'none', 'xtick.labelsize': 6, 'ytick.labelsize': 7, 'legend.frameon': False})
u13_TASK = u17_TASK
u13_ROIS = u17_ROIS
u13_sha = u17_sha
u13_atomic_json = u17_atomic_json
u09_sys.dont_write_bytecode = True
u09_style_ROOT = u09_MRI
u09_sys.dont_write_bytecode = True
u09_brain_CACHE = u09_MRI / 'display_cache'
u09_brain_CACHE.mkdir(exist_ok=True, parents=True)
u09_sys.dont_write_bytecode = True
u09_sys.dont_write_bytecode = True
for u12_key in ('OMP_NUM_THREADS', 'OPENBLAS_NUM_THREADS', 'MKL_NUM_THREADS', 'VECLIB_MAXIMUM_THREADS'):
    u12_os.environ.setdefault(u12_key, '1')
for u12_key in ('OMP_NUM_THREADS', 'OPENBLAS_NUM_THREADS', 'MKL_NUM_THREADS', 'VECLIB_MAXIMUM_THREADS'):
    u12_os.environ.setdefault(u12_key, '1')
u12_group_SYSTEM_FONT = u12_Path('/System/Library/Fonts/Supplemental/Arial.ttf')
if u12_group_SYSTEM_FONT.exists():
    u12_font_manager.fontManager.addfont(u12_group_SYSTEM_FONT)
    u12_plt.rcParams['font.family'] = 'Arial'
u12_brain_CACHE = u12_MRI / 'display_cache'
u12_brain_CACHE.mkdir(exist_ok=True, parents=True)


In [ ]:
def a8_display_three_line(value, *args, **kwargs):
    import pandas as pd
    from IPython.display import display, HTML
    if isinstance(value, pd.DataFrame):
        html = value.to_html(index=False, border=0, escape=True)
        html = html.replace('<table ', '<table style="border-collapse:collapse;border-top:2px solid black;border-bottom:2px solid black;" ', 1)
        html = html.replace('<th>', '<th style="border:0;border-bottom:1px solid black;padding:7px 16px;text-align:center;">')
        html = html.replace('<td>', '<td style="border:0;padding:7px 16px;text-align:center;">')
        return display(HTML(html), *args, **kwargs)
    return display(value, *args, **kwargs)
u09_display = a8_display_three_line


In [ ]:
u09_define = u13_roi_define
u09_define()
u09_display(u09_pd.read_excel(u09_TASK / 'tables/independent_ALE_3D_ROI_definition_5_regions.xlsx', dtype={'subject_id': str, 'subj_idx': str, 'run': str, 'trial_id': str}))


In [ ]:
u09_extract()
u09_display(u09_pd.read_excel(u09_REST / 'tables/functional_connectivity_10_edges_ALE_3D_noGSR.xlsx', dtype={'subject_id': str, 'subj_idx': str, 'run': str, 'trial_id': str}))


In [ ]:
u09_roi_definition()
u09_plot_rest()
for u09_filename in ['Supplementary_resting_ALE_inhibitory_control_ROIs_surface_and_axial_5_regions.png', 'Figure_independent_ALE_ROI_resting_connectivity_matrix_n106.png', 'Figure_independent_ALE_ROI_four_prespecified_connectivity_measures_n106.png']:
    u09_display(u09_Image(filename=str(u09_REST / 'image' / u09_filename), width=850))
